# 第06章 学习的力量 · B1 组（G4）验证

| 被审版本 | 实际小组 | 完整 commit SHA | 文件 |
| --- | --- | --- | --- |
| A1（G12） | G4（本章 B1） | `227b54b200b54af71540079cb4b7aa26013aad3d` | `A1-G12-正文.md` + 4 个实验 notebook + `数据/` |
| A2（G15） | G4（本章 B1） | `1453e18e4d96e65bd8adb88c5806b020e553cf9b` | `A2-G15-正文.md` + `A2组-实验.ipynb` + `A2组-代码-6.3.py` |
| 两稿所在整体版本 | — | `bc956dff468aa2ab1bcaa41e63ab3b253af1ec1d`（`main`） | 同章 B2 组（G9）`工作/B2组-验证报告.md` L16–L18 |

**证据归属（务必区分）**

1. **本 notebook 全部代码单元由 G4 在本轮独立设计并实际执行**，输出为真实 stdout；参考真值来自解析推导或 A1 随稿数据文件（`数据/` 下的只读副本），不复制 A 组的训练结果。
2. A1 / A2 两条既有验证线的执行记录（R1–R5、V1–V8、6.3/6.4 两环境重跑等）**本轮没有重跑**，只在 `B1组-验证报告.md` §3 中作为「引用证据」明确标注来源，不计为本组本轮执行。
3. 本 notebook 不含任何绝对路径、不依赖 NumPy/PyTorch/Jupyter，仅用 CPython 标准库。`数据/` 随包提供 3 个 A1 随稿 CSV 与 1 个 notebook 的**只读副本**；上传到仓库 `第06章/工作/` 后，3 个 CSV 已在同目录 `数据/` 下（**无需重复上传、不要覆盖 A 组文件**），`A1组-实验-6.1.ipynb` 会自动回退到 `工作/` 同级查找。

**执行方式说明（如实记录）**：本机为 CPython 3.13.14（Windows 11），未安装 Jupyter/NumPy。本 notebook 的代码由生成脚本按单元顺序在**同一个 CPython 解释器命名空间**中逐单元 `exec`，stdout 原样保存为各单元的 stream 输出；`execution_count` 是顺序执行编号，**不是 Jupyter 内核计数**。助教在带 Jupyter 的环境可直接 Run All 复核，结果应一致（纯标准库、无随机性）。

**轮次**：后四章初稿阶段第 1 轮验证（2026-11-08 修订）。按 `协作管理/进度安排.md`，后四章 2026-11-30 完成初稿、2026-12-31 完成两次修订，故本轮不称「终版」，A 组修订后由 **B1 复测**并在原问题追加结果。


## 复现环境与参考基准

| 项目 | 内容 |
| --- | --- |
| 环境 | 见 T0 单元打印的实际 `sys.version` / `platform.platform()` |
| 第三方依赖 | **无**。本组对照测试只用 CPython 标准库（math / cmath / csv / json / re / statistics / pathlib） |
| 干净环境口径 | 只安装仓库声明依赖；本组测试不引入任何第三方包，故无 `pip freeze` 差异可产生 |
| 参考真值 | 全部为解析值（√(2/3)、1/√5、π²h²/12、二阶收敛比 4、秩 1、Parseval 尾部能量）或 A1 随稿 CSV / notebook 保存输出 |
| 容差理由 | 解析恒等式用 1e-9～1e-12（浮点舍入量级）；离散格式用 2e-3（h² 以上高阶项）；收敛比用 ±5%（有限网格）；CSV 比对用 2e-3 / 1e-7（随稿数据的记录精度） |
| 未覆盖 | CUDA/float32 路径、torch 训练轨迹、15000 步 Adam、13 张图片、文献原文页码；见报告 §3.3 |


In [1]:
# ===== T0 环境（本组实际执行） =====
import sys, platform, math, cmath, csv, json, re, statistics
from pathlib import Path

print("Python   :", sys.version.split()[0])
print("实现     :", platform.python_implementation(), "/", platform.platform())
print("执行方式 : CPython 逐单元 exec，stdout 原样保存（本机未安装 Jupyter / NumPy）")
third = sorted({m.split('.')[0] for m in sys.modules
                if m.split('.')[0] in {"numpy", "torch", "scipy", "matplotlib", "pandas"}})
print("已加载第三方科学计算模块:", third or "无")
assert not third, "本组对照测试不得依赖第三方包"
DATA = Path("数据")
assert DATA.is_dir(), "缺少 数据/ 目录：请把随稿数据放在 notebook 同级"
print("数据目录文件:", sorted(p.name for p in DATA.iterdir()))
RESULTS = []
print("判据：零第三方依赖、数据齐全 -> 通过")
RESULTS.append(("T0 环境", True))


Python   : 3.13.14
实现     : CPython / Windows-11-10.0.26200-SP0
执行方式 : CPython 逐单元 exec，stdout 原样保存（本机未安装 Jupyter / NumPy）
已加载第三方科学计算模块: 无
数据目录文件: ['A1组-6.2-反问题逐次结果.csv', 'A1组-6.4-反问题逐次结果.csv', 'A1组-6.4-网格结果.csv', 'A1组-实验-6.1.ipynb']
判据：零第三方依赖、数据齐全 -> 通过


## 6.1 PINN：原理与实现（2 学时）

| 被审版本 | 测试输入 | 独立参考值（不取自 A 稿结论） | 容差 | 实测 | 结论 |
| --- | --- | --- | --- | --- | --- |
| A1 / A2 | 只压残差 `-u''=0`，u=2+3x | 离散残差恒 0；核为 span{1,x} | 1e-9 | 见 T1 | 零残差 ≠ 零解 |
| A1 | 加零边界项后解同一方程 | 唯一解 u≡0 | 1e-9 | 见 T1 | 边界项决定唯一性（D1） |
| A1 6.1 表4 / 6.4 表6.4-1 | 中心差分解 `-u''=π²sin(πx)` | max‖e‖=π²h²/12；h 减半误差比 4 | 2e-3 / ±5% | 见 T2 | 二阶收敛成立 |
| A1 随稿 `A1组-6.4-网格结果.csv` FD 行 | rel_l2 / nodal_max | 1/√5 = 0.4472136 | ±2e-3 | 见 T2 | 两处「相对 L2」不是同一范数对象（CH06-G4-B001） |
| A1 6.1 反例三 | ‖1-x‖₂ / ‖sin(πx)‖₂ | √(2/3) = 0.8164966 | 1e-6 | 见 T2 | 正文「约为 1」偏差 18%（CH06-G4-B005） |


In [2]:
# ===== T1 残差零空间与 f≡0 反例（本组独立构造） =====
# 独立预期：-u''=0 的解空间是 span{1,x}；不加边界项时 u=2+3x 是零残差的非零解。
def thomas(a, b, c, d):
    """三对角方程 Ax=d（纯标准库 Thomas 算法）"""
    n = len(d); cp = [0.0]*n; dp = [0.0]*n
    cp[0] = c[0]/b[0]; dp[0] = d[0]/b[0]
    for i in range(1, n):
        m = b[i] - a[i]*cp[i-1]
        cp[i] = (c[i]/m) if i < n-1 else 0.0
        dp[i] = (d[i] - a[i]*dp[i-1])/m
    x = [0.0]*n; x[-1] = dp[-1]
    for i in range(n-2, -1, -1):
        x[i] = dp[i] - cp[i]*x[i+1]
    return x

def fd_residual(u, h):
    return [-(u[i+1] - 2*u[i] + u[i-1])/h**2 for i in range(1, len(u)-1)]

N = 200; h = 1.0/N
u_free = [2.0 + 3.0*(i*h) for i in range(N+1)]
max_r = max(abs(v) for v in fd_residual(u_free, h))
print("只压残差：u=2+3x，max|离散残差| = %.3e，端点 u(0)=%.1f / u(1)=%.1f" % (max_r, u_free[0], u_free[-1]))
assert max_r < 1e-9
assert u_free[0] > 1.0, "与零边界下的精确解 u≡0 应有 O(1) 偏离"

n = N-1
a = [-1.0/h**2]*n; b = [2.0/h**2]*n; c = [-1.0/h**2]*n; d = [0.0]*n
u_bc = thomas(a, b, c, d)
mx = max(abs(v) for v in u_bc)
print("加零边界项：max|u| = %.3e（精确解 u≡0）" % mx)
assert mx < 1e-9
print("判据 1：只压残差时残差为 0；判据 2：但解偏离 u≡0 达 O(1)；判据 3：加边界项后回到零解 -> 全部通过")
print("结论：零损失不等于零解，边界项才使解唯一（独立推导 D1/D2）。")
RESULTS.append(("T1 零空间/f≡0 反例", True))


只压残差：u=2+3x，max|离散残差| = 3.553e-11，端点 u(0)=2.0 / u(1)=5.0
加零边界项：max|u| = 0.000e+00（精确解 u≡0）
判据 1：只压残差时残差为 0；判据 2：但解偏离 u≡0 达 O(1)；判据 3：加边界项后回到零解 -> 全部通过
结论：零损失不等于零解，边界项才使解唯一（独立推导 D1/D2）。


In [3]:
# ===== T2 有限差分解析误差、二阶收敛、范数口径（本组独立复算 + 随稿 CSV 外部核对） =====
def solve_poisson(n):
    h = 1.0/n
    a = [-1.0/h**2]*(n-1); b = [2.0/h**2]*(n-1); c = [-1.0/h**2]*(n-1)
    d = [math.pi**2*math.sin(math.pi*(i*h)) for i in range(1, n)]
    U = [0.0] + thomas(a, b, c, d) + [0.0]
    return max(abs(U[i] - math.sin(math.pi*i*h)) for i in range(n+1))

print("  n     实测 max|e|      pi^2 h^2/12      相对差     与前档误差比")
prev = None
for n in (25, 50, 100, 200, 400):
    e = solve_poisson(n); th = math.pi**2*(1.0/n)**2/12
    ratio = ("%.4f" % (prev/e)) if prev else "   —  "
    print("%4d   %.6e   %.6e   %8.2e   %s" % (n, e, th, abs(e-th)/th, ratio))
    prev = e
e_last = solve_poisson(400); th_last = math.pi**2/400**2/12
assert abs(e_last - th_last)/th_last < 2e-3
ratio2 = solve_poisson(100)/solve_poisson(200)
assert 3.8 < ratio2 < 4.2, ratio2
print("判据：最小网格相对差 <2e-3；h 减半误差比 %.4f ∈ (3.8,4.2) -> 二阶收敛通过" % ratio2)

# 解析范数比（A1 6.1 反例三）
M = 200000
v1 = math.sqrt(sum((1.0-(i+0.5)/M)**2 for i in range(M))/M)   # ||1-x||_2 -> 1/sqrt(3)
v2 = math.sqrt(sum(math.sin(math.pi*(i+0.5)/M)**2 for i in range(M))/M)  # ||sin(pi x)||_2 -> 1/sqrt(2)
q = v1/v2; ref = math.sqrt(2/3)
print("\n||1-x||_2 = %.9f (1/sqrt3=%.9f)，||sin(pi x)||_2 = %.9f (1/sqrt2=%.9f)" %
      (v1, 1/math.sqrt(3), v2, 1/math.sqrt(2)))
print("比值 = %.9f，解析 sqrt(2/3) = %.9f，相对差 %.2e" % (q, ref, abs(q-ref)/ref))
assert abs(q-ref)/ref < 1e-6
print("判据：√(2/3)=0.8164966 复算命中 -> 正文「约为 1」为数学口误（CH06-G4-B005）")

# 外部核对：A1 随稿 CSV
rows = [r for r in csv.DictReader((DATA/"A1组-6.4-网格结果.csv").open(encoding="utf-8-sig"))
        if r["method"] == "FD"]
ratios = [float(r["rel_l2"])/float(r["nodal_max"]) for r in rows]
print("\n随稿 CSV FD 行 n =", [int(r["n"]) for r in rows])
print("rel_l2/nodal_max =", [round(v, 6) for v in ratios], "；1/sqrt(5) =", round(1/math.sqrt(5), 7))
assert all(abs(v - 1/math.sqrt(5)) < 2e-3 for v in ratios)
for r in rows:
    nn = int(r["n"]); nodal = float(r["nodal_max"]); th = math.pi**2/nn**2/12
    assert abs(nodal-th)/th < 2e-3, (nn, nodal, th)
print("判据：%d 行比值均落在 1/√5±2e-3，且 nodal_max = π²h²/12（rtol 2e-3）-> 通过" % len(rows))
print("结论：6.1 表4 填的是节点范数、6.4 表6.4-1 是重建函数范数，比值 1/√5；两者不可混用（CH06-G4-B001 的独立证据）。")
RESULTS.append(("T2 FD 误差/范数口径", True))


  n     实测 max|e|      pi^2 h^2/12      相对差     与前档误差比
  25   1.314388e-03   1.315947e-03   1.18e-03      —  
  50   3.290518e-04   3.289868e-04   1.97e-04   3.9945
 100   8.225076e-05   8.224670e-05   4.93e-05   4.0006
 200   2.056193e-05   2.056168e-05   1.23e-05   4.0001
 400   5.140435e-06   5.140419e-06   3.08e-06   4.0000
判据：最小网格相对差 <2e-3；h 减半误差比 4.0001 ∈ (3.8,4.2) -> 二阶收敛通过

||1-x||_2 = 0.577350269 (1/sqrt3=0.577350269)，||sin(pi x)||_2 = 0.707106781 (1/sqrt2=0.707106781)
比值 = 0.816496581，解析 sqrt(2/3) = 0.816496581，相对差 3.12e-12
判据：√(2/3)=0.8164966 复算命中 -> 正文「约为 1」为数学口误（CH06-G4-B005）

随稿 CSV FD 行 n = [50, 100, 200, 400, 800]
rel_l2/nodal_max = [0.447335, 0.447244, 0.447221, 0.447216, 0.447214] ；1/sqrt(5) = 0.4472136
判据：5 行比值均落在 1/√5±2e-3，且 nodal_max = π²h²/12（rtol 2e-3）-> 通过
结论：6.1 表4 填的是节点范数、6.4 表6.4-1 是重建函数范数，比值 1/√5；两者不可混用（CH06-G4-B001 的独立证据）。


## 6.2 PINN：正问题与反问题（2 学时）

| 被审版本 | 测试输入 | 独立参考值 | 容差 | 实测 | 结论 |
| --- | --- | --- | --- | --- | --- |
| A1 / A2 6.2 | 两参数模型 `-κu''=Aπ²sin(πx)` 的雅可比 | J 两列成比例（比值 −A/κ）、秩 1 | 1e-9 | 见 T3 | 可辨识性仅到 A/κ |
| A1 / A2 6.2 | 沿 (A,κ) 同比例扰动 1% | 精确解不变 | 1e-15 | 见 T3 | 零方向成立 |
| A1 / A2 6.2 | 只动 κ 2% 时的一阶预测 | 出现 O(δ²) 残差 | 残差 >1e-5 且 <线性项 10% | 见 T3 | 线性化有适用范围 |
| A1 6.2 / 6.4 两份 CSV | 25 次反演的 κ̂ | 同源（相对差 <1e-7） | 1e-7 | 见 T3 | 6.4 关于配点网格的文字描述与数据不符 |


In [4]:
# ===== T3 反问题雅可比秩、线性化边界、两 CSV 同源性（本组独立复算） =====
A, kap = 1.0, 2.0
xd = [j/6 for j in range(1, 6)]
def u_two(Av, kv, x): return (Av/kv)*math.sin(math.pi*x)
def J1(x): return math.sin(math.pi*x)/kap
def J2(x): return -A*math.sin(math.pi*x)/kap**2

h = 1e-6; diffs = []
for x in xd:
    fd1 = (u_two(A+h, kap, x) - u_two(A-h, kap, x))/(2*h)
    fd2 = (u_two(A, kap+h, x) - u_two(A, kap-h, x))/(2*h)
    diffs += [abs(fd1-J1(x)), abs(fd2-J2(x))]
print("解析雅可比 vs 中心有限差最大差 = %.3e" % max(diffs))
assert max(diffs) < 1e-9
props = [J2(x)/J1(x) for x in xd]
print("col2/col1 =", [round(v, 9) for v in props], "；理论 -A/κ =", -A/kap)
assert all(abs(v + A/kap) < 1e-12 for v in props)
print("判据：两列成比例 -> 秩 1，只有 a=A/κ 可辨识")

d0 = (0.01*A, 0.01*kap)
ex = max(abs(u_two(A+d0[0], kap+d0[1], x) - u_two(A, kap, x)) for x in xd)
print("沿 (A,κ) 同比例扰动 1%% 的精确解变化 = %.3e" % ex)
assert ex < 1e-15

dk = 0.02*kap
lin = sum(J2(x)*dk for x in xd)/len(xd)
exact = sum(u_two(A, kap+dk, x) - u_two(A, kap, x) for x in xd)/len(xd)
print("只动 κ 2%%：一阶预测 %.6e，精确 %.6e，二阶残差 %.3e（占线性项 %.2f%%）" %
      (lin, exact, exact-lin, 100*abs(exact-lin)/abs(lin)))
assert abs(exact-lin) > 1e-5 and abs(exact-lin) < abs(lin)*0.1
print("判据：一阶线性化只在小扰动成立，δ² 项可测 -> 通过")

d62 = {(r["setting"], r["seed"]): float(r["kappa_hat"]) for r in
       csv.DictReader((DATA/"A1组-6.2-反问题逐次结果.csv").open(encoding="utf-8-sig"))}
d64 = {(r["setting"], r["seed"]): float(r["kappa_hat"]) for r in
       csv.DictReader((DATA/"A1组-6.4-反问题逐次结果.csv").open(encoding="utf-8-sig"))
       if r["method"] == "PINN-inverse" and r["kappa_hat"]}
keys = sorted(set(d62) & set(d64))
mx = max(abs(d62[k]-d64[k])/abs(d62[k]) for k in keys)
print("\n两 CSV 共同键 %d 个（setting,seed），κ̂ 最大相对差 = %.3e" % (len(keys), mx))
assert len(keys) >= 20 and mx < 1e-7
print("判据：25 次反演结果同源到 1e-7 以下 -> 6.4 的配点网格文字描述与数据不符（CH06-G4-B013 侧证）")
RESULTS.append(("T3 反问题/线性化", True))


解析雅可比 vs 中心有限差最大差 = 2.416e-11
col2/col1 = [-0.5, -0.5, -0.5, -0.5, -0.5] ；理论 -A/κ = -0.5
判据：两列成比例 -> 秩 1，只有 a=A/κ 可辨识
沿 (A,κ) 同比例扰动 1% 的精确解变化 = 0.000e+00
只动 κ 2%：一阶预测 -7.464102e-03，精确 -7.317747e-03，二阶残差 1.464e-04（占线性项 1.96%）
判据：一阶线性化只在小扰动成立，δ² 项可测 -> 通过

两 CSV 共同键 25 个（setting,seed），κ̂ 最大相对差 = 4.729e-08
判据：25 次反演结果同源到 1e-7 以下 -> 6.4 的配点网格文字描述与数据不符（CH06-G4-B013 侧证）


## 6.3 DeepONet 与 FNO（2 学时）

| 被审版本 | 测试输入 | 独立参考值 | 容差 | 实测 | 结论 |
| --- | --- | --- | --- | --- | --- |
| A1 / A2 6.3 | 谱层 `F⁻¹(R F x)` vs 显式循环卷积 | 逐点相等 | 1e-10 | 见 T4 | FNO 谱层即循环卷积 |
| A1 / A2 6.3 | M 个模态 × C 通道 | 实参数量 2MC² | 恒等 | 见 T4 | 参数随 C² 增长 |
| A1 / A2 6.3 | 非周期 f(x)=x 截断 \|k\|≤4 | 边界误差 > 3× 内部 | 比值 >3 | 见 T4 | 边界 Gibbs 伪影 |
| A1 6.3 / A2 6.3 | 循环谱算子的截断逼近 | 误差 = 被舍弃模态的尾部能量（Parseval） | 1e-8 | 见 T5 | 秩 p 下界成立 |
| A1 / A2 6.3 | K≤4 输入子空间 | 自由度 1+2K = 9 | 恒等 | 见 T5 | p≥9 才能精确 |

> 本组**没有**重跑 15000 步 Adam 训练；A2 正文 9.36e-2 / 1.60e-1 / 1.19e-1 的出处问题（CH06-G4-B004，P0）仍以两线既有执行记录为证据，见报告 §3.2，本 notebook 不复现该训练。


In [5]:
# ===== T4 谱层=循环卷积、参数量、非周期 Gibbs（本组独立实现 DFT） =====
def dft(x):
    N = len(x)
    return [sum(x[j]*cmath.exp(-2j*math.pi*k*j/N) for j in range(N)) for k in range(N)]
def idft(X):
    N = len(X)
    return [sum(X[k]*cmath.exp(2j*math.pi*k*j/N) for k in range(N))/N for j in range(N)]
def kfreq(k, N): return k if k <= N//2 else k-N

N, M = 32, 4
R = [cmath.exp(-0.05*abs(kfreq(k, N))) if abs(kfreq(k, N)) <= M else 0j for k in range(N)]
xin = [math.sin(2*math.pi*3*j/N) + 0.5*math.cos(2*math.pi*7*j/N) for j in range(N)]
X = dft(xin)
y_spec = [v.real for v in idft([R[k]*X[k] for k in range(N)])]
kern = [v.real for v in idft(R)]
y_circ = [sum(xin[j]*kern[(i-j) % N] for j in range(N)) for i in range(N)]
d = max(abs(a-b) for a, b in zip(y_spec, y_circ))
print("谱层输出 vs 显式循环卷积 max|diff| = %.3e" % d)
assert d < 1e-10
C = 3; n_real = 2*M*C*C
print("参数量：2*M*C² = 2*%d*%d² = %d" % (M, C, n_real))
assert n_real == 2*M*C*C

N2 = 64
x2 = [j/N2 for j in range(N2)]
F = dft(x2)
rec = [v.real for v in idft([F[k] if abs(kfreq(k, N2)) <= 4 else 0j for k in range(N2)])]
err = [abs(a-b) for a, b in zip(rec, x2)]
edge = max(err[:8] + err[-8:]); mid = max(err[16:48])
print("非周期 f(x)=x 截断 |k|<=4：边界 max err=%.4f，内部 max err=%.4f，比值=%.2f" %
      (edge, mid, edge/mid))
assert edge > 3*mid
print("判据：循环卷积等价、参数量恒等、边界伪影显著 -> 全部通过")
RESULTS.append(("T4 FNO 谱层/Gibbs", True))


谱层输出 vs 显式循环卷积 max|diff| = 5.995e-15
参数量：2*M*C² = 2*4*3² = 72
非周期 f(x)=x 截断 |k|<=4：边界 max err=0.4297，内部 max err=0.0403，比值=10.66
判据：循环卷积等价、参数量恒等、边界伪影显著 -> 全部通过


In [6]:
# ===== T5 截断谱算子的误差下界（Parseval，纯标准库） =====
N = 64
ks = [kfreq(k, N) for k in range(N)]
mult = [1.0/(1.0 + (2*math.pi*k)**2) for k in ks]

def W_entry(dd, mask):
    return sum(mult[k]*math.cos(2*math.pi*ks[k]*dd/N)
               for k in range(N) if mask[k])/N
def frob(mask):
    # 循环矩阵每行是 W_entry(d)，共 N 行
    return math.sqrt(N*sum(W_entry(d, mask)**2 for d in range(N)))

den = frob([True]*N)
print("  m   实秩   数值 Frobenius 相对误差   解析尾部能量比     相对差")
for m in (0, 1, 2, 4, 8):
    keep = [abs(k) <= m for k in ks]
    drop = [not v for v in keep]
    e_num = frob(drop)/den
    e_ana = math.sqrt(sum(v*v for v, k in zip(mult, ks) if abs(k) > m)/
                      sum(v*v for v in mult))
    print("%3d  %5d      %.9e      %.9e   %.2e" % (m, 2*m+1, e_num, e_ana, abs(e_num-e_ana)))
    assert abs(e_num-e_ana) < 1e-8
print("判据：数值 Frobenius 误差 = 解析尾部能量（Parseval），容差 1e-8 -> 通过")
K = 4; dof = 1 + 2*K
print("K<=%d 输入子空间自由度 = 1+2*%d = %d，故秩 p>=%d 才能在该子空间上精确" % (K, K, dof, dof))
assert dof == 9
RESULTS.append(("T5 秩 p 下界/子空间秩", True))


  m   实秩   数值 Frobenius 相对误差   解析尾部能量比     相对差
  0      1      3.637834255e-02      3.637834255e-02   6.94e-18
  1      3      1.021627370e-02      1.021627370e-02   0.00e+00
  2      5      5.027950317e-03      5.027950317e-03   8.67e-19
  4      9      2.134743443e-03      2.134743443e-03   4.34e-19
  8     17      8.231058750e-04      8.231058750e-04   1.08e-19
判据：数值 Frobenius 误差 = 解析尾部能量（Parseval），容差 1e-8 -> 通过
K<=4 输入子空间自由度 = 1+2*4 = 9，故秩 p>=9 才能在该子空间上精确


## 6.4 AI 关联：科学基础模型（2 学时）

| 被审版本 | 测试输入 | 独立参考值 | 容差 | 实测 | 结论 |
| --- | --- | --- | --- | --- | --- |
| A1 6.1 notebook 保存输出 | 7 段 `t_total`（保序去重） | 逐次和 = 末尾 total = 68.335592 s | 1e-6 | 见 T6 | notebook 自洽 |
| A1 6.1 正文 L524 | 正文「全 Notebook 训练总计 241.2 s」 | 与 total 之比 3.53 | 3.5–3.6 | 见 T6 | 两套时间不同源（CH06-G4-B002，P0） |
| A1 6.2/6.4 配点网格 | 中心网格 vs 含端点等距网格 | 是否含 x=0、残差 RMS 相对差 | 结构判据 | 见 T7 | 网格定义影响残差指标 |
| A2 6.4 | 表 6.4-4 的 40 个表值 | 两线既有记录 40/40 命中 | 报告 §3.2 引用 | **本轮未重跑** | 引用证据，不计为本组执行 |

> 6.4 的科学基础模型属文献综述，本组不验证论文结论本身；只核对可确定性复算的部分。


In [7]:
# ===== T6 时间口径：从 A1 随稿 notebook 的保存输出独立验算（本组未重跑训练） =====
nb_path = DATA/"A1组-实验-6.1.ipynb"                 # 交付包内副本
if not nb_path.exists():
    nb_path = Path("A1组-实验-6.1.ipynb")             # 仓库 工作/ 目录（与 notebook 同级）
assert nb_path.exists(), "未找到 A1组-实验-6.1.ipynb：请放在 数据/ 或 notebook 同级"
print("读取 A1 6.1 保存输出：", nb_path)
nb = json.loads(nb_path.read_text(encoding="utf-8"))
chunks = []
for c in nb["cells"]:
    for o in c.get("outputs", []):
        if o.get("output_type") == "stream":
            t = o.get("text", ""); chunks.append(t if isinstance(t, str) else "".join(t))
        elif "data" in o and "text/plain" in o["data"]:
            t = o["data"]["text/plain"]; chunks.append(t if isinstance(t, str) else "".join(t))
blob = "\n".join(chunks)
tt_all = [float(v) for v in re.findall(r'"t_total":\s*([0-9.eE+\-]+)', blob)]
tt = []
for v in tt_all:                      # 保序去重：λ_b=1 那行复用了 seed0 的 t_total
    if v not in tt: tt.append(v)
total = float(re.search(r"total training time \(s\):\s*([0-9.]+)", blob).group(1))
paper = 241.2
print("保存输出中 t_total 出现 %d 次，保序去重后 %d 次" % (len(tt_all), len(tt)))
print("去重后逐次和 = %.6f s；notebook 末尾 total = %.6f s；差 = %.3e" %
      (sum(tt), total, abs(sum(tt)-total)))
assert len(tt) == 7 and abs(sum(tt)-total) < 1e-6
print("正文 241.2 s / total = %.4f 倍（3.53）" % (paper/total))
assert 3.5 < paper/total < 3.6
print("判据：同一份保存输出内部自洽；正文 241.2 s 与 notebook total 不同源（CH06-G4-B002，P0）")
print("说明：本组只解析保存输出并验算加和，**没有**重跑 6.1 训练。")
RESULTS.append(("T6 时间口径", True))


读取 A1 6.1 保存输出： 数据\A1组-实验-6.1.ipynb
保存输出中 t_total 出现 8 次，保序去重后 7 次
去重后逐次和 = 68.335592 s；notebook 末尾 total = 68.335592 s；差 = 0.000e+00
正文 241.2 s / total = 3.5296 倍（3.53）
判据：同一份保存输出内部自洽；正文 241.2 s 与 notebook total 不同源（CH06-G4-B002，P0）
说明：本组只解析保存输出并验算加和，**没有**重跑 6.1 训练。


In [8]:
# ===== T7 配点网格定义对残差指标的影响（本组独立构造） =====
def resid(x):
    # 候选边界层 u(x)=sin(pi x)+0.05 exp(-x/0.02) 的 -u'' 与方程右端之差
    return -125.0*math.exp(-x/0.02)

xc = [(i+0.5)/256 for i in range(256)]     # 单元中心（6.2 的说法）
xe = [i/255 for i in range(256)]           # 含端点等距（6.4 的说法）
r_c = math.sqrt(sum(resid(x)**2 for x in xc)/len(xc))
r_e = math.sqrt(sum(resid(x)**2 for x in xe)/len(xe))
rel = abs(r_e-r_c)/r_c
print("中心网格含 x=0？", 0.0 in xc, "；含端点网格含 x=0？", 0.0 in xe)
print("残差 RMS：中心网格 = %.6f，含端点网格 = %.6f，相对差 = %.2f%%" %
      (r_c, r_e, 100*rel))
assert (0.0 in xe) and (0.0 not in xc)
assert rel > 0.05, rel
print("判据：两网格结构不同，残差 RMS 相差 %.2f%%（>5%%）-> 指标确实依赖配点定义" % (100*rel))
print("结论：6.2/6.4 两份 CSV 的 25 次结果同源（T3），但 6.4 关于配点网格的文字描述需修正。")
RESULTS.append(("T7 配点网格", True))


中心网格含 x=0？ False ；含端点网格含 x=0？ True
残差 RMS：中心网格 = 12.460377，含端点网格 = 13.716659，相对差 = 10.08%
判据：两网格结构不同，残差 RMS 相差 10.08%（>5%）-> 指标确实依赖配点定义
结论：6.2/6.4 两份 CSV 的 25 次结果同源（T3），但 6.4 关于配点网格的文字描述需修正。


## 复现结论与修正清单

- 本组本轮**实际执行**了 T0–T7 共 8 个代码单元，覆盖 6.1–6.4 的解析可判定部分；全部断言通过、无 error。
- **引用但本轮未重跑**：A1 线 R1–R5 / V1–V8 / 6.2 的 25 次重跑、A2 线 6.3/6.4 两环境执行与表 6.4-4 的 40/40 复算。这些是 A1/A2 两条既有验证线的记录，见 `B1组-验证报告.md` §3，不计为本组本轮执行。
- 问题编号见 `B1组-修正清单.md`（`CH06-G4-B001`–`B045`）；A 组修复后由 **B1 复测**并在原问题追加「复测结论」。


In [9]:
# ===== 汇总：所有判据集中复核（本组实际执行） =====
print("%-26s %s" % ("单元", "结果"))
print("-"*40)
for name, ok in RESULTS:
    print("%-26s %s" % (name, "PASS" if ok else "FAIL"))
print("-"*40)
assert len(RESULTS) == 8, "缺少单元：%d" % len(RESULTS)
assert all(ok for _, ok in RESULTS), "存在未通过的判据"
print("全部 %d 个独立对照测试通过；0 error；零第三方依赖。" % len(RESULTS))
print("参考真值汇总：√(2/3)=%.7f，1/√5=%.7f，π²h²/12(h=0.02)=%.4e，"
      "241.2/68.335592=%.4f" % (math.sqrt(2/3), 1/math.sqrt(5),
                                math.pi**2*0.02**2/12, 241.2/68.335592))


单元                         结果
----------------------------------------
T0 环境                      PASS
T1 零空间/f≡0 反例              PASS
T2 FD 误差/范数口径              PASS
T3 反问题/线性化                 PASS
T4 FNO 谱层/Gibbs            PASS
T5 秩 p 下界/子空间秩             PASS
T6 时间口径                    PASS
T7 配点网格                    PASS
----------------------------------------
全部 8 个独立对照测试通过；0 error；零第三方依赖。
参考真值汇总：√(2/3)=0.8164966，1/√5=0.4472136，π²h²/12(h=0.02)=3.2899e-04，241.2/68.335592=3.5296
